In [26]:
import hashlib
import hmac
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Rutas del proyecto
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = RAIZ / "datos" / "raw"
INTERIM_DIR = RAIZ / "datos" / "interim"
CLEAN_DIR = RAIZ / "datos" / "clean"
OUTPUT_DIR = RAIZ / "output"

for d in (RAW_DIR, INTERIM_DIR, CLEAN_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Configuración de scraping responsable
BASE_URL = "https://corunacorre.com"
EDICION_ID = "63"
HEADERS = {"User-Agent": "P2-CorunaCorre-BigData-UAC/1.0 (practica academica)"}
PAUSE = 1.5

archivo_env = RAIZ / ".env"
assert archivo_env.is_file(), "Falta el archivo .env en la raíz del repositorio."
variables = dict(
    linea.split("=", 1)
    for linea in archivo_env.read_text(encoding="utf-8").splitlines()
    if "=" in linea and not linea.lstrip().startswith("#")
)
CLAVE_HMAC = variables.get("CORUNA_HMAC_KEY", "").strip().encode("utf-8")
assert CLAVE_HMAC, "Falta CORUNA_HMAC_KEY en .env."



def generar_id(clave_persona: str) -> str:
    return hmac.new(CLAVE_HMAC, clave_persona.encode("utf-8"), hashlib.sha256).hexdigest()[:12]

def normalizar_nombre(serie: pd.Series) -> pd.Series:
    return (
        serie.fillna("")
        .str.normalize("NFKD")
        .str.replace(r"[\u0300-\u036f]", "", regex=True)
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Cargar datasets limpios e intermedios necesarios para el enlace local
participacions = pd.read_csv(CLEAN_DIR / "participacions.csv")
carreiras = pd.read_csv(CLEAN_DIR / "carreiras.csv").sort_values("orde")
inscritos = pd.read_csv(INTERIM_DIR / "inscritos_circuito_interim.csv", dtype="string", keep_default_na=False)

participacions = participacions.merge(
    carreiras[["id_carrera", "orde", "nombre_carrera"]],
    on="id_carrera",
    how="left",
    validate="many_to_one",
)

print(f"Participaciones cargadas: {len(participacions)}")
print(f"Carreras disputadas: {len(carreiras)}")

Participaciones cargadas: 5390
Carreras disputadas: 6


In [27]:
def descargar_pagina(url: str, nombre_archivo: str) -> bytes:
    """Descarga una página con User-Agent y pausa >= 1.5s, usando caché en datos/raw/."""
    ruta = RAW_DIR / nombre_archivo
    if ruta.exists():
        print(f"  [CACHÉ] {nombre_archivo}")
        return ruta.read_bytes()

    print(f"  [DESCARGA] {url}")
    time.sleep(PAUSE)
    resp = requests.get(url, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    ruta.write_bytes(resp.content)
    return resp.content

# URL base proporcionada para la clasificación del circuito
URL_CLASIF_PRINCIPAL = f"{BASE_URL}/es/web/resultado/edicion-{EDICION_ID}/clasificacion/157"

# Descargar página 1 y comprobar si en la propia página hay pestañas/enlaces a otras clasificaciones (ej. otro sexo/categoría)
html_p1 = descargar_pagina(f"{URL_CLASIF_PRINCIPAL}?page=1", "clasif_157_pag_01.html")
soup_p1 = BeautifulSoup(html_p1, "lxml")

enlaces_clasificacion = [{"id": "157", "texto": "CLASIF_157", "url": URL_CLASIF_PRINCIPAL}]
ids_vistos = {"157"}

for a in soup_p1.find_all("a", href=True):
    m = re.search(rf"/resultado/edicion-{EDICION_ID}/clasificacion/(\d+)", a["href"])
    if m and m.group(1) not in ids_vistos:
        cid = m.group(1)
        texto = a.get_text(" ", strip=True) or f"CLASIF_{cid}"
        if not any(ex in texto.upper() for ex in ["PITUFOS", "BENXAMIN", "BENJAMÍN", "ALEVIN", "ALEVÍN", "INFANTIL", "CADETE", "XUVENIL", "JUVENIL"]):
            ids_vistos.add(cid)
            enlaces_clasificacion.append({
                "id": cid,
                "texto": texto,
                "url": f"{BASE_URL}/es/web/resultado/edicion-{EDICION_ID}/clasificacion/{cid}",
            })

print("Clasificaciones a descargar:")
for e in enlaces_clasificacion:
    print(f"  - [{e['id']}] {e['texto']} -> {e['url']}")

  [CACHÉ] clasif_157_pag_01.html
Clasificaciones a descargar:
  - [157] CLASIF_157 -> https://corunacorre.com/es/web/resultado/edicion-63/clasificacion/157


In [28]:
def parsear_tabla_clasificacion(html_bytes: bytes, etiqueta_clasif: str, num_pagina: int) -> list[dict]:
    soup = BeautifulSoup(html_bytes, "lxml")
    tabla = soup.find("table")
    if not tabla:
        return []

    ths = [th.get_text(" ", strip=True).lower() for th in tabla.find_all("th")]
    tbody = tabla.find("tbody")
    trs = tbody.find_all("tr") if tbody else tabla.find_all("tr")[1:]

    filas = []
    for tr in trs:
        tds = [td.get_text(" ", strip=True) for td in tr.find_all("td")]
        if len(tds) < 4:
            continue

        registro = {
            "clasificacion_origen": etiqueta_clasif,
            "pagina": num_pagina,
        }
        if ths and len(ths) == len(tds):
            for col_name, val in zip(ths, tds):
                registro[col_name] = val
        else:
            offset = 1 if tds[0] == "" else 0
            campos_pos = ["puesto_web", "dorsal", "nombre", "apellidos", "detalle_cat", "puntos_o_pruebas"]
            for i, campo in enumerate(campos_pos):
                if offset + i < len(tds):
                    registro[campo] = tds[offset + i]
        filas.append(registro)
    return filas

filas_web_totales = []

for item in enlaces_clasificacion:
    cid = item["id"]
    etiqueta = item["texto"]
    url_base = item["url"]

    print(f"\nDescargando clasificación {cid} ({etiqueta})...")
    pagina = 1
    huella_previa = set()

    while True:
        url_pag = f"{url_base}?page={pagina}"
        archivo_raw = f"clasif_{cid}_pag_{pagina:02d}.html"

        html = descargar_pagina(url_pag, archivo_raw)
        filas = parsear_tabla_clasificacion(html, etiqueta, pagina)

        if not filas:
            (RAW_DIR / archivo_raw).unlink(missing_ok=True)
            break

        huella_actual = {tuple(f.values()) for f in filas}
        if huella_actual == huella_previa:
            (RAW_DIR / archivo_raw).unlink(missing_ok=True)
            break

        huella_previa = huella_actual
        filas_web_totales.extend(filas)

        if len(filas) < 25:
            break
        pagina += 1

df_web_raw = pd.DataFrame(filas_web_totales)
print(f"\nTotal registros descargados de la clasificación web: {len(df_web_raw)}")
print("Columnas detectadas (sin mostrar datos personales):", list(df_web_raw.columns))


Descargando clasificación 157 (CLASIF_157)...
  [CACHÉ] clasif_157_pag_01.html
  [CACHÉ] clasif_157_pag_02.html
  [CACHÉ] clasif_157_pag_03.html
  [CACHÉ] clasif_157_pag_04.html
  [CACHÉ] clasif_157_pag_05.html
  [CACHÉ] clasif_157_pag_06.html
  [CACHÉ] clasif_157_pag_07.html
  [CACHÉ] clasif_157_pag_08.html
  [CACHÉ] clasif_157_pag_09.html
  [CACHÉ] clasif_157_pag_10.html
  [CACHÉ] clasif_157_pag_11.html
  [CACHÉ] clasif_157_pag_12.html
  [CACHÉ] clasif_157_pag_13.html
  [CACHÉ] clasif_157_pag_14.html
  [CACHÉ] clasif_157_pag_15.html

Total registros descargados de la clasificación web: 352
Columnas detectadas (sin mostrar datos personales): ['clasificacion_origen', 'pagina', 'puesto', 'nombre', 'apellidos', 'puntos', 'desempate1', 'puesto sexo', 'puesto categoria', 'nº pruebas']


In [29]:
web_proc = df_web_raw.copy()

# 1. Normalizar nombre completo y cruzar con el padrón de inscritos para obtener el mismo persoa_id
nombre_full = web_proc["nombre"].fillna("") + " " + web_proc["apellidos"].fillna("")
web_proc["nombre_normalizado"] = normalizar_nombre(nombre_full)

inscritos["nombre_normalizado"] = normalizar_nombre(inscritos["nombre_completo"])
inscritos["dorsal"] = inscritos["dorsal"].fillna("").str.strip()

dorsales_por_nombre = inscritos.groupby("nombre_normalizado")["dorsal"].nunique()
mapa_dorsal_unico = (
    inscritos[inscritos["nombre_normalizado"].isin(dorsales_por_nombre[dorsales_por_nombre == 1].index)]
    .set_index("nombre_normalizado")["dorsal"]
    .to_dict()
)

def asignar_clave_sin_dorsal(nom: str) -> str:
    if nom in mapa_dorsal_unico:
        return f"padron:{mapa_dorsal_unico[nom]}:{nom}"
    return f"nombre:{nom}"

web_proc["clave_persona"] = web_proc["nombre_normalizado"].map(asignar_clave_sin_dorsal)
web_proc["persoa_id"] = web_proc["clave_persona"].map(generar_id)

# 2. Parsear columnas numéricas de la clasificación oficial
def a_entero(serie: pd.Series) -> pd.Series:
    return pd.to_numeric(serie.astype(str).str.extract(r"(\d+)", expand=False), errors="coerce").astype("Int64")

web_proc["puesto_general_web"] = a_entero(web_proc["puesto"])
web_proc["puesto_sexo_web"] = a_entero(web_proc["puesto sexo"])
web_proc["puesto_categoria_web"] = a_entero(web_proc["puesto categoria"])
web_proc["puntos_web"] = pd.to_numeric(
    web_proc["puntos"].astype(str).str.replace(",", "."), errors="coerce"
)
web_proc["desempate1_web"] = web_proc["desempate1"].fillna("").str.strip()
web_proc["carreras_web"] = a_entero(web_proc["nº pruebas"])

# Guardar copia local en interim/ y quedarnos en memoria SOLO con columnas pseudonimizadas
web_proc.to_csv(INTERIM_DIR / "clasificacion_web_local.csv", index=False)

clasif_web = web_proc[[
    "persoa_id",
    "clasificacion_origen",
    "puesto_general_web",
    "puesto_sexo_web",
    "puesto_categoria_web",
    "puntos_web",
    "desempate1_web",
    "carreras_web",
]].copy()

print(f"Clasificación web pseudonimizada: {len(clasif_web)} filas, {clasif_web['persoa_id'].nunique()} personas únicas.")
display(clasif_web.head(10))

Clasificación web pseudonimizada: 352 filas, 352 personas únicas.


,persoa_id,clasificacion_origen,puesto_general_web,puesto_sexo_web,puesto_categoria_web,puntos_web,desempate1_web,carreras_web
0,ae0e0933f424,CLASIF_157,1,1,14,6.0,"5,00",<NA>
1,60efcab4a192,CLASIF_157,2,2,18,8.0,"5,00",<NA>
2,e568b1fcae60,CLASIF_157,3,1,12,12.0,"5,00",<NA>
3,d5590e691954,CLASIF_157,4,3,14,18.0,"6,00",<NA>
4,ab1072c5e1b0,CLASIF_157,5,4,14,18.0,"5,00",<NA>
5,fd997f1f617e,CLASIF_157,6,2,10,18.0,"5,00",<NA>
6,73864d7f25ca,CLASIF_157,7,5,18,20.0,"5,00",<NA>
7,6b27bc657bee,CLASIF_157,8,6,1,20.0,"5,00",<NA>
8,cb101ec9c309,CLASIF_157,9,7,16,27.0,"6,00",<NA>
9,7bc521299683,CLASIF_157,10,3,14,27.0,"6,00",<NA>


In [30]:
SOLO_INSCRITOS_CIRCUITO = True   # En Coruña Corre solo puntúan los inscritos en el circuito
REORDENAR_PUESTOS_SIN_POR_PRUEBA = False  # False: suma el puesto real de cada carrera sin recalcular
AMBITO_PUNTOS = "general"        # "general" (puesto_general, el de CLASIF_157), "sexo" o "categoria"
MEJORES_N_CARRERAS = 5           # Se computan los 5 mejores resultados
MIN_CARRERAS_PARA_CLASIFICAR = 5 # Mínimo reglamentario de 5 pruebas completadas

df_calc = participacions.copy()

# 1. Filtrar participaciones válidas e inscritos en el circuito
df_calc = df_calc[df_calc["estado_tiempo"].isin(["valido", "revisar"])].copy()
if SOLO_INSCRITOS_CIRCUITO:
    df_calc = df_calc[df_calc["tipo_inscripcion"] == "inscrito_circuito"].copy()

# 2. Seleccionar columna de puesto base según el ámbito
mapa_col_puesto = {
    "general": "puesto_general",
    "sexo": "puesto_por_sexo",
    "categoria": "puesto_por_categoria",
}
col_puesto_base = mapa_col_puesto[AMBITO_PUNTOS]

df_calc = df_calc.dropna(subset=["sexo", "categoria", col_puesto_base]).copy()

if REORDENAR_PUESTOS_SIN_POR_PRUEBA:
    cols_grupo = ["id_carrera"] if AMBITO_PUNTOS == "general" else (
        ["id_carrera", "sexo"] if AMBITO_PUNTOS == "sexo" else ["id_carrera", "sexo", "categoria"]
    )
    df_calc = df_calc.sort_values(cols_grupo + [col_puesto_base, "tiempo_segundos"])
    df_calc["puntos_carrera"] = df_calc.groupby(cols_grupo).cumcount() + 1
else:
    df_calc["puntos_carrera"] = df_calc[col_puesto_base].astype(int)

# 3. Ordenar carreras de cada persona de mejor (menor puesto) a peor
df_calc = df_calc.sort_values(["persoa_id", "puntos_carrera"], ascending=[True, True])
df_calc["rank_carrera_persoa"] = df_calc.groupby("persoa_id").cumcount() + 1

# 4. Resumen por persona y suma de las 5 mejores carreras
resumen_persoa = (
    df_calc.groupby("persoa_id", as_index=False)
    .agg(
        sexo=("sexo", "first"),
        categoria=("categoria", "last"),
        carreras_disputadas=("id_carrera", "nunique"),
        tiempo_total_seg=("tiempo_segundos", "sum"),
    )
)

mejores = df_calc[df_calc["rank_carrera_persoa"] <= MEJORES_N_CARRERAS]
puntos_persoa = (
    mejores.groupby("persoa_id", as_index=False)
    .agg(
        carreras_computadas=("id_carrera", "nunique"),
        puntos_calculados=("puntos_carrera", "sum"),
        mejor_puesto=("puntos_carrera", "min"),
    )
)

clasif_calculada = resumen_persoa.merge(puntos_persoa, on="persoa_id", how="inner")

# 5. Filtrar por mínimo de 5 carreras
clasif_final_calc = clasif_calculada[
    clasif_calculada["carreras_disputadas"] >= MIN_CARRERAS_PARA_CLASIFICAR
].copy()

# 6. Ordenar y calcular puesto general, puesto por sexo y puesto por categoría (como en la web)
clasif_final_calc = clasif_final_calc.sort_values(
    ["puntos_calculados", "carreras_disputadas", "mejor_puesto", "tiempo_total_seg"],
    ascending=[True, False, True, True]
).reset_index(drop=True)

clasif_final_calc["puesto_general_calc"] = clasif_final_calc.index + 1
clasif_final_calc["puesto_sexo_calc"] = clasif_final_calc.groupby("sexo").cumcount() + 1
clasif_final_calc["puesto_categoria_calc"] = clasif_final_calc.groupby(["sexo", "categoria"]).cumcount() + 1
clasif_final_calc["puesto_calculado"] = clasif_final_calc["puesto_general_calc"]

print(f"Total corredores inscritos que puntuaron en al menos 1 carrera: {len(clasif_calculada)}")
print(f"Total corredores que cumplen el mínimo ({MIN_CARRERAS_PARA_CLASIFICAR} carreras): {len(clasif_final_calc)}")
display(clasif_final_calc.head(15))

Total corredores inscritos que puntuaron en al menos 1 carrera: 1206
Total corredores que cumplen el mínimo (5 carreras): 292


,persoa_id,sexo,categoria,carreras_disputadas,tiempo_total_seg,carreras_computadas,puntos_calculados,mejor_puesto,puesto_general_calc,puesto_sexo_calc,puesto_categoria_calc,puesto_calculado
0,6b27bc657bee,M,MS1,5,6571,5,20,2,1,1,1,1
1,d2fc7426be7a,M,MS2,6,7934,5,27,3,2,2,1,2
2,21c1606afc21,M,MS1,6,7969,5,28,4,3,3,2,3
3,dcc0c17acfb7,M,MS1,5,6646,5,33,2,4,4,3,4
4,487fd8809a6a,M,SNR,5,6641,5,33,5,5,5,1,5
5,fc0e7788acf1,M,SUB20,6,8218,5,54,9,6,6,1,6
6,d362dc14a3bf,M,SUB20,5,6883,5,54,7,7,7,2,7
7,54fe1f24731f,M,SNR,5,6797,5,57,7,8,8,2,8
8,84e68907a5a2,M,SNR,6,8411,5,74,11,9,9,3,9
9,2b17b7265b76,M,SNR,5,6883,5,74,8,10,10,4,10


In [31]:
col_puesto_web_ref = "puesto_general_web" if AMBITO_PUNTOS == "general" else (
    "puesto_sexo_web" if AMBITO_PUNTOS == "sexo" else "puesto_categoria_web"
)
clasif_web_unica = (
    clasif_web.sort_values(col_puesto_web_ref)
    .drop_duplicates(subset=["persoa_id"], keep="first")
    .rename(columns={col_puesto_web_ref: "puesto_web"})
)

comparacion = clasif_final_calc.merge(
    clasif_web_unica,
    on="persoa_id",
    how="outer",
    indicator=True
)

# Enriquecer con información diagnóstica de participacions (sin nombres)
info_diag = (
    participacions.groupby("persoa_id", as_index=False)
    .agg(
        tipo_inscripcion=("tipo_inscripcion", "first"),
        resultado_cruce=("resultado_cruce", "first"),
        total_participaciones=("id_carrera", "count"),
        tiene_cat_NO=("categoria", lambda s: (s == "NO").any()),
        tiene_tiempo_revisar=("estado_tiempo", lambda s: (s != "valido").any()),
    )
)
comparacion = comparacion.merge(info_diag, on="persoa_id", how="left")

# Clasificar el resultado de la comparación persona a persona
def diagnosticar_caso(row):
    if row["_merge"] == "both":
        diff_pts = abs((row["puntos_calculados"] or 0) - (row["puntos_web"] or 0))
        diff_pos = abs((row["puesto_calculado"] or 0) - (row["puesto_web"] or 0))
        if diff_pts == 0 and diff_pos == 0:
            return "1. Coincidencia exacta (puesto y puntos)"
        elif diff_pts == 0 and diff_pos > 0:
            return "2. Mismos puntos, distinto puesto (criterio de desempate)"
        else:
            return "3. En ambas clasificaciones, pero con diferencia de puntos"
    elif row["_merge"] == "left_only":
        return "4. Solo en nuestro cálculo (no aparece en la web)"
    else:
        if row.get("tipo_inscripcion") == "no_resuelto":
            return "5a. Solo en la web: caso 'no_resuelto' en nuestro enlace"
        elif row.get("tipo_inscripcion") == "por_prueba":
            return "5b. Solo en la web: clasificado como 'por_prueba' en nuestro cruce"
        elif pd.notna(row.get("total_participaciones")) and row["total_participaciones"] < MIN_CARRERAS_PARA_CLASIFICAR:
            return "5c. Solo en la web: menos carreras de nuestro mínimo"
        else:
            return "5d. Solo en la web: otra causa (dorsal/nombre distinto)"

comparacion["diagnostico"] = comparacion.apply(diagnosticar_caso, axis=1)

resumen_diff = comparacion["diagnostico"].value_counts().to_frame("num_personas")
resumen_diff["porcentaje"] = (resumen_diff["num_personas"] / len(comparacion) * 100).round(2)

print("=== RESUMEN DE LA COMPARACIÓN PERSONA A PERSONA ===")
display(resumen_diff)

# Inspeccionar muestras pseudonimizadas de los casos con diferencias para su explicación
print("\n=== MUESTRA DE CASOS CON DIFERENCIA DE PUNTOS O PUESTO (SIN DATOS PERSONALES) ===")
cols_muestra = [
    "persoa_id", "sexo", "categoria", "carreras_disputadas", "carreras_computadas",
    "puntos_calculados", "puntos_web", "puesto_calculado", "puesto_web", "diagnostico"
]
display(comparacion[comparacion["_merge"] == "both"].query("puntos_calculados != puntos_web or puesto_calculado != puesto_web")[cols_muestra].head(10))

# Exportar resultado anonimizado a output/
comparacion.drop(columns=["_merge"]).to_csv(OUTPUT_DIR / "o4_comparacion_clasificacion.csv", index=False)
print(f"\nComparación pseudonimizada exportada en: {OUTPUT_DIR / 'o4_comparacion_clasificacion.csv'}")

=== RESUMEN DE LA COMPARACIÓN PERSONA A PERSONA ===


,num_personas,porcentaje
diagnostico,,
"2. Mismos puntos, distinto puesto (criterio de desempate)",290,82.39
5d. Solo en la web: otra causa (dorsal/nombre distinto),50,14.20
5c. Solo en la web: menos carreras de nuestro mínimo,10,2.84
"3. En ambas clasificaciones, pero con diferencia de puntos",2,0.57



=== MUESTRA DE CASOS CON DIFERENCIA DE PUNTOS O PUESTO (SIN DATOS PERSONALES) ===


,persoa_id,sexo,categoria,carreras_disputadas,carreras_computadas,puntos_calculados,puntos_web,puesto_calculado,puesto_web,diagnostico
1,011ae44b638b,F,SNR,5.0,5.0,4602.0,4602.0,292.0,352,"2. Mismos puntos, distinto puesto (criterio de..."
2,01264947b830,M,MS4,6.0,5.0,3482.0,3482.0,255.0,313,"2. Mismos puntos, distinto puesto (criterio de..."
3,0171413bb1af,F,SUB23,6.0,5.0,3395.0,3395.0,253.0,311,"2. Mismos puntos, distinto puesto (criterio de..."
4,01ce9cddc27f,M,MS2,5.0,5.0,101.0,101.0,12.0,49,"2. Mismos puntos, distinto puesto (criterio de..."
5,0338163f7289,F,SNR,6.0,5.0,1826.0,1826.0,158.0,212,"2. Mismos puntos, distinto puesto (criterio de..."
6,0366288fa5b8,M,MS3,6.0,5.0,2045.0,2045.0,176.0,232,"2. Mismos puntos, distinto puesto (criterio de..."
7,042a0cdabf6c,M,MS3,5.0,5.0,1029.0,1029.0,103.0,156,"2. Mismos puntos, distinto puesto (criterio de..."
8,055236ee8d70,M,MS1,6.0,5.0,589.0,589.0,61.0,111,"2. Mismos puntos, distinto puesto (criterio de..."
9,0580f7a2085f,F,MS3,5.0,5.0,4346.0,4346.0,285.0,345,"2. Mismos puntos, distinto puesto (criterio de..."
10,070d3692553a,F,SNR,5.0,5.0,2327.0,2327.0,194.0,250,"2. Mismos puntos, distinto puesto (criterio de..."



Comparación pseudonimizada exportada en: /Users/sergio/Desktop/Clase/davi/p2-coruna-corre/output/o4_comparacion_clasificacion.csv
